In [2]:
from pathlib import Path
import pandas as pd
import geopandas as gpd

ROOT = Path(".").resolve()
PG_DIR = ROOT / "powergenome_wecc_solar_2030"
RESOURCE_DIR = PG_DIR / "resource_groups"
SITE_MAP_DIR = PG_DIR / "site_maps"
COUNTY_GEOJSON = ROOT / "county_layer_for_gui.geojson"
OUTPUT_DIR = PG_DIR / "outputs"
OUTPUT_DIR.mkdir(exist_ok=True)

solar_df = pd.read_parquet(RESOURCE_DIR / "solar_lcoe_wecc_solar_2030.parquet")
wecc14_df = pd.read_csv(SITE_MAP_DIR / "solar_lcoe_wecc_14_zone.csv", low_memory=False)

wecc14_df = wecc14_df.rename(columns={"CPA_ID": "cpa_id"})

print("solar_df shape:", solar_df.shape)
print("wecc14_df shape:", wecc14_df.shape)
print(wecc14_df.columns.tolist())

FileNotFoundError: [Errno 2] No such file or directory: '/Users/laurenvo/Documents/github/solar-county-propensity-scores/notebooks/powergenome_wecc_solar_2030/outputs'

In [ ]:
solar_ids = set(solar_df["cpa_id"].dropna().unique())
wecc14_ids = set(wecc14_df["cpa_id"].dropna().unique())
overlap = solar_ids & wecc14_ids

print("solar unique ids:", len(solar_ids))
print("wecc14 unique ids:", len(wecc14_ids))
print("overlap ids:", len(overlap))
print("solar match rate:", len(overlap) / len(solar_ids))

solar unique ids: 159408
wecc14 unique ids: 402688
overlap ids: 157172
solar match rate: 0.9859731004717455


In [ ]:
merged = solar_df.merge(
    wecc14_df[["cpa_id", "longitude", "latitude"]],
    on="cpa_id",
    how="left"
)

print("missing lon share:", merged["longitude"].isna().mean())
print("missing lat share:", merged["latitude"].isna().mean())
display(merged.head())

missing lon share: 0.014026899528254541
missing lat share: 0.014026899528254541


,cpa_id,msa_id,cpa_mw,cf,path,path_len,lcoe,interconnect_capex_mw,total_interconnect_km,region,msa_name,anyQual,m_popden,exFacil,plFacil,capacity_mw,longitude,latitude
0,219261,19700,34.747166,0.319071,"[219261,300657]",2.0,22.620781,17228.830078,1.103553,NM1,"Deming, NM",1,5.757576,0,0,34.747166,-107.791199,32.304256
1,212932,37220,89.863327,0.330012,"[212932,310177]",2.0,22.628979,63279.906250,15.071069,NV1,"Pahrump, NV",1,0.000000,0,0,89.863327,-115.831334,36.174793
2,107220,29420,111.978813,0.318505,"[107220,303516]",2.0,22.663147,17356.224609,0.957107,AZ1,"Lake Havasu City-Kingman, AZ",1,0.276596,0,0,111.978813,-114.141078,35.060884
3,219262,19700,37.833187,0.319071,"[219262,300657]",2.0,22.682138,20832.226562,2.707107,NM1,"Deming, NM",1,9.210526,0,0,37.833187,-107.753521,32.310589
4,212941,37220,70.248444,0.330012,"[212941,310177]",2.0,22.682661,66540.640625,16.399496,NV1,"Pahrump, NV",1,0.000000,0,0,70.248444,-115.846086,36.245512


In [ ]:
counties = gpd.read_file(COUNTY_GEOJSON).to_crs("EPSG:4326").copy()
counties = counties.rename(columns={
    "NAME": "county_name",
    "STATE_NAME": "state_name",
    "STUSPS": "state_abbrev",
    "GEOID": "county_fips",
})

sites_gdf = gpd.GeoDataFrame(
    merged,
    geometry=gpd.points_from_xy(merged["longitude"], merged["latitude"]),
    crs="EPSG:4326"
)

sites_with_counties = gpd.sjoin(
    sites_gdf,
    counties[["county_name", "state_name", "state_abbrev", "county_fips", "geometry"]],
    how="left",
    predicate="within"
)

print("missing county_fips share:", sites_with_counties["county_fips"].isna().mean())
display(sites_with_counties.head())

missing county_fips share: 0.014026899528254541


,cpa_id,msa_id,cpa_mw,cf,path,path_len,lcoe,interconnect_capex_mw,total_interconnect_km,region,...,capacity_mw,longitude,latitude,geometry,index_right,county_name,county_name,state_name,state_abbrev,county_fips
0,219261,19700,34.747166,0.319071,"[219261,300657]",2.0,22.620781,17228.830078,1.103553,NM1,...,34.747166,-107.791199,32.304256,POINT (-107.7912 32.30426),431.0,Luna,None,New Mexico,NM,35029
1,212932,37220,89.863327,0.330012,"[212932,310177]",2.0,22.628979,63279.906250,15.071069,NV1,...,89.863327,-115.831334,36.174793,POINT (-115.83133 36.17479),799.0,Clark,None,Nevada,NV,32003
2,107220,29420,111.978813,0.318505,"[107220,303516]",2.0,22.663147,17356.224609,0.957107,AZ1,...,111.978813,-114.141078,35.060884,POINT (-114.14108 35.06088),6.0,Mohave,None,Arizona,AZ,04015
3,219262,19700,37.833187,0.319071,"[219262,300657]",2.0,22.682138,20832.226562,2.707107,NM1,...,37.833187,-107.753521,32.310589,POINT (-107.75352 32.31059),431.0,Luna,None,New Mexico,NM,35029
4,212941,37220,70.248444,0.330012,"[212941,310177]",2.0,22.682661,66540.640625,16.399496,NV1,...,70.248444,-115.846086,36.245512,POINT (-115.84609 36.24551),799.0,Clark,None,Nevada,NV,32003


In [ ]:
counties = gpd.read_file(COUNTY_GEOJSON).to_crs("EPSG:4326").copy()
counties = counties.rename(columns={
    "NAME": "county_name_geo",
    "STATE_NAME": "state_name_geo",
    "STUSPS": "state_abbrev_geo",
    "GEOID": "county_fips_geo",
})

sites_gdf = gpd.GeoDataFrame(
    merged.copy(),
    geometry=gpd.points_from_xy(merged["longitude"], merged["latitude"]),
    crs="EPSG:4326"
)

sites_with_counties = gpd.sjoin(
    sites_gdf,
    counties[["county_name_geo", "state_name_geo", "state_abbrev_geo", "county_fips_geo", "geometry"]],
    how="left",
    predicate="within"
)

print("missing county_fips share:", sites_with_counties["county_fips_geo"].isna().mean())
display(sites_with_counties.head())

missing county_fips share: 0.014026899528254541


,cpa_id,msa_id,cpa_mw,cf,path,path_len,lcoe,interconnect_capex_mw,total_interconnect_km,region,...,plFacil,capacity_mw,longitude,latitude,geometry,index_right,county_name_geo,state_name_geo,state_abbrev_geo,county_fips_geo
0,219261,19700,34.747166,0.319071,"[219261,300657]",2.0,22.620781,17228.830078,1.103553,NM1,...,0,34.747166,-107.791199,32.304256,POINT (-107.7912 32.30426),431.0,Luna,New Mexico,NM,35029
1,212932,37220,89.863327,0.330012,"[212932,310177]",2.0,22.628979,63279.906250,15.071069,NV1,...,0,89.863327,-115.831334,36.174793,POINT (-115.83133 36.17479),799.0,Clark,Nevada,NV,32003
2,107220,29420,111.978813,0.318505,"[107220,303516]",2.0,22.663147,17356.224609,0.957107,AZ1,...,0,111.978813,-114.141078,35.060884,POINT (-114.14108 35.06088),6.0,Mohave,Arizona,AZ,04015
3,219262,19700,37.833187,0.319071,"[219262,300657]",2.0,22.682138,20832.226562,2.707107,NM1,...,0,37.833187,-107.753521,32.310589,POINT (-107.75352 32.31059),431.0,Luna,New Mexico,NM,35029
4,212941,37220,70.248444,0.330012,"[212941,310177]",2.0,22.682661,66540.640625,16.399496,NV1,...,0,70.248444,-115.846086,36.245512,POINT (-115.84609 36.24551),799.0,Clark,Nevada,NV,32003


In [ ]:
county_candidate_summary = (
    sites_with_counties.groupby(
        ["state_abbrev_geo", "state_name_geo", "county_name_geo", "county_fips_geo"],
        dropna=False
    )
    .agg(
        candidate_site_count=("cpa_id", "count"),
        total_candidate_mw=("capacity_mw", "sum"),
        avg_lcoe=("lcoe", "mean"),
        min_lcoe=("lcoe", "min"),
    )
    .reset_index()
    .rename(columns={
        "state_abbrev_geo": "state_abbrev",
        "state_name_geo": "state_name",
        "county_name_geo": "county_name",
        "county_fips_geo": "county_fips",
    })
    .sort_values(["avg_lcoe", "total_candidate_mw"], ascending=[True, False])
)

county_candidate_summary.to_csv(
    OUTPUT_DIR / "powergenome_candidate_solar_by_county.csv",
    index=False
)

sites_with_counties.drop(columns="geometry").to_csv(
    OUTPUT_DIR / "powergenome_candidate_solar_sites_with_counties.csv",
    index=False
)

display(county_candidate_summary.head(20))

,state_abbrev,state_name,county_name,county_fips,candidate_site_count,total_candidate_mw,avg_lcoe,min_lcoe
461,TX,Texas,El Paso,48141,96,9199.701172,24.140999,22.936644
357,NV,Nevada,Storey,32029,40,2916.803711,24.234159,23.647406
310,NM,New Mexico,Bernalillo,35001,138,13883.432617,24.347578,23.251474
342,NM,New Mexico,Valencia,35061,173,18370.226562,25.189569,23.898010
317,NM,New Mexico,Doña Ana,35013,453,37938.351562,25.303665,22.817089
336,NM,New Mexico,Santa Fe,35049,256,22689.408203,25.394381,23.648464
335,NM,New Mexico,Sandoval,35043,492,53655.648438,25.659870,23.038004
11,AZ,Arizona,Pinal,04021,815,73097.093750,25.743645,23.541727
306,NE,Nebraska,Scotts Bluff,31157,136,14921.298828,25.851501,24.928759
325,NM,New Mexico,Los Alamos,35028,9,283.266998,25.866213,25.514870


In [ ]:
print("solar match rate:", len(overlap) / len(solar_ids))
print("missing county_fips share:", sites_with_counties["county_fips_geo"].isna().mean())

solar match rate: 0.9859731004717455
missing county_fips share: 0.014026899528254541
